In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install dagshub
!pip install mlflow

In [5]:
import sys
sys.path.append('/kaggle/usr/lib/notebooks/nikolozdodashvili/preprocessing/notebooks/nikolozdodashvili')

from preprocessing import (
    optimize_memory,
    DropHighMissingFeatures,
    FrequencyEncoder,
    MissingValueImputer,
    TimeFeatureExtractor,
    TransactionAmtTransformer,
    GroupAggregator,
    DropCorrelatedFeatures
)

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud'].astype('int8')

X = optimize_memory(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

მეხსიერება დამუშავებამდე: 1946.36 MB
მეხსიერება დამუშავების შემდეგ: 921.47 MB
შემცირდა: 52.7%


In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.frequency_maps = {}

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            self.frequency_maps[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col, freq_map in self.frequency_maps.items():
            if col in X.columns:
                if hasattr(X[col], 'cat'):
                    X[col] = X[col].astype('object')
                X[col] = X[col].map(freq_map).fillna(0)
        return X

# TRAIN AND MLFLOW

In [9]:
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

dagshub.init(repo_owner='ndoda23',
             repo_name='MachineLearning---IEEE-CIS-Fraud-Detection',
             mlflow=True)

mlflow.set_experiment("LogisticRegression_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a5054a0a-4df2-495c-bdcc-17ff356488f3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=bceb4ececf0a1e80b9c1dd107904d017d25b11f39e1ef95571a4f24bc2e35a13




Accessing as ndoda23

Initialized MLflow to track repo "ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection"

Repository ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection initialized!

2026/05/06 16:26:32 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ce40c4ffc9cf4c859a5516bcbe6aca4f', creation_time=1778084792775, experiment_id='3', last_update_time=1778084792775, lifecycle_stage='active', name='LogisticRegression_Training', tags={}, trace_location=None, workspace='default'>

In [10]:
with mlflow.start_run(run_name="LR_Baseline_Run"):
    print("Starting Logistic Regression Baseline Run...")
    
    pipeline_baseline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('scaler',   StandardScaler()),  
        ('model',    LogisticRegression(max_iter=1000, random_state=42))
    ])

    pipeline_baseline.fit(X_train, y_train)

    y_train_proba = pipeline_baseline.predict_proba(X_train)[:, 1]
    y_val_proba = pipeline_baseline.predict_proba(X_val)[:, 1]
    y_val_class = pipeline_baseline.predict(X_val)
    
    train_auc = roc_auc_score(y_train, y_train_proba)
    val_auc = roc_auc_score(y_val, y_val_proba)

    print("Running Cross-Validation for Baseline...")
    cv_scores = cross_val_score(pipeline_baseline, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

    mlflow.log_params({"model_type": "LogisticRegression", "C": 1.0, "penalty": "l2"})
    mlflow.log_metrics({
        "train_roc_auc": train_auc,
        "val_roc_auc": val_auc,
        "train_val_diff": train_auc - val_auc,
        "cv_auc_mean": cv_scores.mean(),
        "val_f1": f1_score(y_val, y_val_class)
    })
    
    mlflow.sklearn.log_model(pipeline_baseline, "lr_baseline_model")
    print(f"✅ Baseline Completed! Val AUC: {val_auc:.4f}")



Starting Logistic Regression Baseline Run...
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Running Cross-Validation for Baseline...


2026/05/06 16:29:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:29:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline Completed! Val AUC: 0.8567
🏃 View run LR_Baseline_Run at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/6511c249fc8a4b438f80241528aa36c8
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3


In [11]:

with mlflow.start_run(run_name="LR_Optimized_Run"):
    print("\nStarting Logistic Regression Optimized Run...")
    
    pipeline_optimized = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('scaler',   StandardScaler()),
        ('model',    LogisticRegression(
            C=0.1,                  
            class_weight='balanced', 
            max_iter=1000, 
            random_state=42
        ))
    ])

    pipeline_optimized.fit(X_train, y_train)

    y_train_proba_opt = pipeline_optimized.predict_proba(X_train)[:, 1]
    y_val_proba_opt = pipeline_optimized.predict_proba(X_val)[:, 1]
    y_val_class_opt = pipeline_optimized.predict(X_val)
    
    train_auc_opt = roc_auc_score(y_train, y_train_proba_opt)
    val_auc_opt = roc_auc_score(y_val, y_val_proba_opt)

    print("Running Cross-Validation for Optimized...")
    cv_scores_opt = cross_val_score(pipeline_optimized, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

    mlflow.log_params({"C": 0.1, "class_weight": "balanced", "max_iter": 1000})
    mlflow.log_metrics({
        "train_roc_auc": train_auc_opt,
        "val_roc_auc": val_auc_opt,
        "train_val_diff": train_auc_opt - val_auc_opt,
        "cv_auc_mean": cv_scores_opt.mean(),
        "val_f1": f1_score(y_val, y_val_class_opt),
        "val_recall": recall_score(y_val, y_val_class_opt) # Recall მნიშვნელოვანია ბალანსირებისას
    })
    
    mlflow.sklearn.log_model(pipeline_optimized, "lr_optimized_model")
    print(f"✅ Optimized Completed! Val AUC: {val_auc_opt:.4f}")


Starting Logistic Regression Optimized Run...
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Running Cross-Validation for Optimized...
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420


2026/05/06 16:35:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:35:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Optimized Completed! Val AUC: 0.8632
🏃 View run LR_Optimized_Run at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/fcc91e0af3594884ac34fa6023e859ff
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
